# 01 · Issuers, ratings and the news feed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammedelsayed741996-blip/berlin-data-portfolio/blob/main/consulting-financial-services/credit-downgrade-early-warning/notebooks/01_data_and_news.ipynb)

The question a bank or investor asks: which of the companies we lend to are likely to be downgraded soon, and can the news tell us before the rating agency does?

All data here is synthetic. 360 made-up European issuers in ten sectors, with ratings from AA to CCC, quarterly financials, share prices for the listed ones, and about 66,000 news articles in English and German from January 2022 to September 2026. No article names a person, and no company is real.

In [1]:
import os, sys, subprocess, warnings
from pathlib import Path

REPO = "https://github.com/mohammedelsayed741996-blip/berlin-data-portfolio.git"
PROJECT = "consulting-financial-services/credit-downgrade-early-warning"

if "google.colab" in sys.modules:
    if not Path("/content/berlin-data-portfolio").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/berlin-data-portfolio"], check=True)
    os.chdir(f"/content/berlin-data-portfolio/{PROJECT}")
else:
    here = Path.cwd()
    while not (here / "src" / "downgrade_ews").exists() and here != here.parent:
        here = here.parent
    os.chdir(here)

sys.path.insert(0, str(Path.cwd() / "src"))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 110)
from IPython.display import Image, display
print("Project folder:", Path.cwd().name)

Project folder: credit-downgrade-early-warning


In [2]:
from downgrade_ews import config as C, pipeline, features, models, explain, validation, reporting

tables = pipeline.load_tables()      # reads data/, or generates it if missing
run = pipeline.run(tables)           # text model, panel, four issuer models (about a minute)
print(f"{len(tables['news']):,} articles on {C.N_ISSUERS} issuers, {C.START} to {C.END}")

65,908 articles on 360 issuers, 2022-01 to 2026-09


## The issuers

In [3]:
iss = tables["issuers"]
display(iss.head(8))
display(iss.groupby("sector").size().rename("issuers").to_frame().T)

,issuer_id,issuer_name,sector,country,listed,size,start_rating
0,ISS0001,Veneon Handel SpA,Retail,IT,False,0.92,BBB
1,ISS0002,Ardund Property SA,Real estate,FR,False,0.55,BBB
2,ISS0003,Linax Technik AG,Industrials,DE,False,0.31,BB+
3,ISS0004,Wesora Property BV,Real estate,NL,False,0.43,BB+
4,ISS0005,Calen Industrie GmbH,Industrials,DE,True,0.76,BB
5,ISS0006,Steova Wohnen GmbH,Real estate,DE,False,2.33,A+
6,ISS0007,Valeon Drive AG,Automotive,DE,True,0.63,BB
7,ISS0008,Norvalt Power SA,Utilities,FR,True,1.39,AA-


sector,Automotive,Chemicals,Consumer goods,Healthcare,Industrials,Real estate,Retail,Telecoms,Transport & logistics,Utilities
issuers,31,38,23,37,43,44,44,31,31,38


## How credit quality and ratings move

Each issuer has a hidden credit score in rating notches. It moves with sector shocks (real estate after the rate rise in 2022, chemicals on gas prices, automotive in 2024/25), slow deterioration that builds over four to eight months, and rare sudden events. The agency rating follows that score with a lag of one to three months, and outlook changes only come when the agency happens to review the name. That lag is the window an early-warning system can use.

In [4]:
a = tables["rating_actions"]
display(a["action"].value_counts().to_frame().T)
dg = a[a["action"] == "downgrade"]
dg.groupby(dg["month"].str[:4]).size().rename("downgrades per year").to_frame().T

action,downgrade,outlook negative,outlook positive,upgrade,outlook stable
count,266,205,138,109,109


month,2022,2023,2024,2025,2026
downgrades per year,43,67,50,60,46


## What the news looks like

Articles come from templates, but the mix of topics follows each company's hidden path. A company sliding toward a downgrade gets more refinancing stories and profit warnings. Healthy companies get bad headlines too, troubled ones put out upbeat releases, some articles deny problems, and sector stories mention a company without being about it.

In [5]:
n = tables["news"]
display(n.sample(8, random_state=7)[["date", "language", "source", "headline"]])
print(n["language"].value_counts(normalize=True).round(2).to_dict())

{'en': 0.66, 'de': 0.34}


,date,language,source,headline
38991,2024-10-23,en,Continental Business Review,Briund Health SA publishes sustainability report
56957,2026-02-06,en,Euro Credit Wire,Korova Energie BV faces lawsuit over supply contract
56153,2026-01-17,de,Finanzblatt,Telara Stores AG verkauft Sparte und senkt Schulden
38944,2024-10-22,en,Continental Business Review,Solex Connect SAS books EUR 400 million impairment
16735,2023-03-19,en,Continental Business Review,Zanica Materials GmbH appoints new supervisory board member
5492,2022-06-03,de,Finanzblatt,Hauptversammlung von Steora Chemie AG stimmt allen Punkten zu
52044,2025-10-01,en,Euro Credit Wire,Haven Property GmbH publishes sustainability report
57579,2026-02-21,de,Finanzblatt,Hauptversammlung von Eldund Consumer AG stimmt allen Punkten zu


In [6]:
lat = tables["latent"][["issuer_id", "month", "stress_gap", "momentum_3m"]]
mix = n.assign(month=n["date"].str[:7]).merge(lat, on=["issuer_id", "month"])
mix["state"] = pd.cut(mix["stress_gap"] + mix["momentum_3m"], [-99, 0.3, 1.0, 99], labels=["calm", "drifting", "stressed"])
top = mix.groupby("state")["true_topic"].value_counts(normalize=True).unstack().T
top.loc[["profit_warning", "refinancing", "covenant", "liquidity", "denial", "contract_win", "agm"]].round(3)

state,calm,drifting,stressed
true_topic,,,
profit_warning,0.016,0.032,0.068
refinancing,0.005,0.021,0.067
covenant,0.003,0.014,0.045
liquidity,0.004,0.016,0.042
denial,0.016,0.031,0.067
contract_win,0.060,0.052,0.030
agm,0.134,0.108,0.057


## The target

For every issuer and month: is the issuer downgraded in the next three months? Features use only what was known at the end of that month: articles dated in or before it, financials already published (two months after quarter end), past share prices. The last three months have no label yet.

Training runs to March 2025. The test covers July 2025 to June 2026. The three months in between are left out, so no training label looks into the test period.

In [7]:
p = run.panel
print(f"Labelled issuer-months: {p['labelled'].sum():,}, downgrade rate {p.loc[p['labelled'], 'y'].mean():.1%}")
print(f"Train {len(run.train):,} rows up to {run.train['month'].max()}, test {len(run.test):,} rows from {run.test['month'].min()}")

Labelled issuer-months: 19,440, downgrade rate 3.9%
Train 12,930 rows up to 2025-03, test 4,310 rows from 2025-07


Before the text reaches a model, the company name, sector, city, quarter and every number are masked, and the words after a negation are marked. Notebook 03 explains why each of these was needed.

In [8]:
ex = run.scored.sample(3, random_state=3)
for h, t in zip(ex["headline"], ex["text"]):
    print(h); print("  ->", t[:160]); print()

Hauptversammlung von Ardeon Chemie AG stimmt allen Punkten zu
  -> hauptversammlung von [co] stimmt allen punkten zu. die aktionäre von [co] haben auf der hauptversammlung allen beschlussvorschlägen zugestimmt.

healthcare sector faces weaker demand; Rovion Medical SA says it is not affected
  -> [sector] sector faces weaker demand; [co] says it is not neg_affected. several [sector] companies have warned on profits this month. [co] said its order book re

Zanyn Drive NV suspends dividend to preserve cash
  -> [co] suspends dividend to preserve cash. [co] will not neg_pay neg_a neg_dividend neg_for [year]. the board said protecting the balance sheet had priority.

